# ANN vs LSTM with Dropout & Early Stopping
Complete Practical Notebook with Comparison

In [ ]:
!pip install numpy pandas scikit-learn tensorflow matplotlib

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.callbacks import EarlyStopping


## Create Time Series Data

In [ ]:

data = np.sin(np.arange(0, 100, 0.1))

def create_dataset(data, time_step=10):
    X, y = [], []
    for i in range(len(data)-time_step-1):
        X.append(data[i:(i+time_step)])
        y.append(data[i+time_step])
    return np.array(X), np.array(y)

time_step = 10
X, y = create_dataset(data, time_step)

X_lstm = X.reshape(X.shape[0], X.shape[1], 1)

split = int(0.8 * len(X))
X_train, X_test = X_lstm[:split], X_lstm[split:]
y_train, y_test = y[:split], y[split:]


## LSTM Model

In [ ]:

lstm_model = Sequential()
lstm_model.add(LSTM(50, return_sequences=True, input_shape=(time_step,1)))
lstm_model.add(Dropout(0.3))
lstm_model.add(LSTM(50))
lstm_model.add(Dropout(0.3))
lstm_model.add(Dense(1))

lstm_model.compile(optimizer='adam', loss='mse')

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_lstm = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop]
)

lstm_loss = lstm_model.evaluate(X_test, y_test)
print("LSTM Loss:", lstm_loss)


## ANN Model

In [ ]:

X_ann = X.reshape(X.shape[0], X.shape[1])
X_train_ann, X_test_ann = X_ann[:split], X_ann[split:]

ann_model = Sequential()
ann_model.add(Dense(64, activation='relu', input_dim=X_train_ann.shape[1]))
ann_model.add(Dropout(0.3))
ann_model.add(Dense(32, activation='relu'))
ann_model.add(Dropout(0.3))
ann_model.add(Dense(1))

ann_model.compile(optimizer='adam', loss='mse')

early_stop_ann = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_ann = ann_model.fit(
    X_train_ann, y_train,
    validation_data=(X_test_ann, y_test),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop_ann]
)

ann_loss = ann_model.evaluate(X_test_ann, y_test)
print("ANN Loss:", ann_loss)


## Comparison Table

In [ ]:

results = pd.DataFrame({
    "Model": ["ANN", "LSTM"],
    "Test Loss": [ann_loss, lstm_loss]
})

print(results)


## Loss Graph

In [ ]:

plt.plot(history_lstm.history['loss'], label='LSTM Train')
plt.plot(history_lstm.history['val_loss'], label='LSTM Val')

plt.plot(history_ann.history['loss'], label='ANN Train')
plt.plot(history_ann.history['val_loss'], label='ANN Val')

plt.legend()
plt.title("Loss Comparison")
plt.show()
